In [2]:
# -----------------------------
# FULL CORRECTED ViT (MNIST) CODE
# Shows: batch loss+acc during training and epoch summary (train+test)
# -----------------------------

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torch.utils.data as dataloader

# -----------------------
# Device
# -----------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# -----------------------
# Data
# -----------------------
transform = transforms.Compose([transforms.ToTensor()])

train_dataset = torchvision.datasets.MNIST(
    root="data", train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.MNIST(
    root="data", train=False, download=True, transform=transform
)

batch_size = 64
train_loader = dataloader.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = dataloader.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# -----------------------
# Hyperparameters
# -----------------------
num_classes = 10
img_size = 28
num_channels = 1
patch_size = 7
patch_num = (img_size // patch_size) ** 2  # 16

attention_heads = 4
embedding_dim = 128
num_transformer_layers = 4
mlp_dim = 256
dropout_rate = 0.1
mlp_nodes = 128

# -----------------------
# Model Components
# -----------------------
class PatchEmbedding(nn.Module):
    def __init__(self, num_channels, embedding_dim, patch_size):
        super().__init__()
        self.patch_embed = nn.Conv2d(
            in_channels=num_channels,
            out_channels=embedding_dim,
            kernel_size=patch_size,
            stride=patch_size
        )

    def forward(self, x):
        # x: (B, C, H, W)
        x = self.patch_embed(x)                 # (B, E, H/p, W/p) => (B, E, 4, 4)
        x = x.flatten(2).transpose(1, 2)        # (B, 16, E)
        return x

class TransformerBlock(nn.Module):
    def __init__(self, embedding_dim, attention_heads, mlp_dim, dropout_rate):
        super().__init__()
        self.norm1 = nn.LayerNorm(embedding_dim)
        self.attn = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=attention_heads,
            dropout=dropout_rate,
            batch_first=True  # IMPORTANT: expects (B, T, E)
        )
        self.norm2 = nn.LayerNorm(embedding_dim)

        self.mlp = nn.Sequential(
            nn.Linear(embedding_dim, mlp_dim),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(mlp_dim, embedding_dim),
            nn.Dropout(dropout_rate),
        )

    def forward(self, x):
        # x: (B, T, E)
        residual = x
        x = self.norm1(x)
        x, _ = self.attn(x, x, x)  # self-attention
        x = x + residual

        residual2 = x
        x = self.norm2(x)
        x = self.mlp(x)
        x = x + residual2
        return x

class MLPHead(nn.Module):
    def __init__(self, embedding_dim, mlp_nodes, num_classes, dropout_rate):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(embedding_dim, mlp_nodes),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(mlp_nodes, num_classes)
        )

    def forward(self, x):
        # x: (B, T, E)
        x = x[:, 0]  # CLS token
        return self.mlp(x)

class ViT(nn.Module):
    def __init__(self):
        super().__init__()
        self.patch_embed = PatchEmbedding(num_channels, embedding_dim, patch_size)

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embedding_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, patch_num + 1, embedding_dim))

        self.blocks = nn.ModuleList([
            TransformerBlock(embedding_dim, attention_heads, mlp_dim, dropout_rate)
            for _ in range(num_transformer_layers)
        ])

        self.head = MLPHead(embedding_dim, mlp_nodes, num_classes, dropout_rate)

        # init
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)

    def forward(self, x):
        x = self.patch_embed(x)                 # (B, 16, E)
        B = x.size(0)

        cls = self.cls_token.expand(B, -1, -1)  # (B, 1, E)
        x = torch.cat([cls, x], dim=1)          # (B, 17, E)

        x = x + self.pos_embed                  # (B, 17, E)

        for blk in self.blocks:
            x = blk(x)

        x = self.head(x)                        # (B, num_classes)
        return x

# -----------------------
# Train / Eval helpers
# -----------------------
def batch_accuracy(logits, y):
    preds = logits.argmax(dim=1)
    return (preds == y).float().mean().item()

def train_one_epoch(model, loader, optimizer, criterion, device, epoch, epochs, log_every=200):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for batch_idx, (x, y) in enumerate(loader, start=1):
        x, y = x.to(device), y.to(device)

        logits = model(x)
        loss = criterion(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # batch stats
        b_loss = loss.item()
        b_acc = batch_accuracy(logits, y)

        total_loss += b_loss * y.size(0)
        total_correct += (logits.argmax(dim=1) == y).sum().item()
        total_samples += y.size(0)

        # show batch loss + batch accuracy
        if (batch_idx % log_every == 0) or (batch_idx == 1) or (batch_idx == len(loader)):
            print(f"Epoch [{epoch}/{epochs}] "
                  f"Batch [{batch_idx}/{len(loader)}] "
                  f"Loss: {b_loss:.4f} | Acc: {b_acc*100:.2f}%")

    epoch_loss = total_loss / total_samples
    epoch_acc = total_correct / total_samples
    return epoch_loss, epoch_acc

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)

        total_loss += loss.item() * y.size(0)
        total_correct += (logits.argmax(dim=1) == y).sum().item()
        total_samples += y.size(0)

    return total_loss / total_samples, total_correct / total_samples

# -----------------------
# Train
# -----------------------
model = ViT().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

epochs = 10
log_every = 200     # set 1 if you want every batch printed

for epoch in range(1, epochs + 1):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, criterion, device,
        epoch=epoch, epochs=epochs, log_every=log_every
    )

    test_loss, test_acc = evaluate(model, test_loader, criterion, device)

    print(f"\n=== Epoch {epoch} Summary ===")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}%")
    print(f"Test  Loss: {test_loss:.4f} | Test  Acc: {test_acc*100:.2f}%\n")

Using device: cuda
Epoch [1/10] Batch [1/938] Loss: 2.2923 | Acc: 17.19%
Epoch [1/10] Batch [200/938] Loss: 0.9332 | Acc: 71.88%
Epoch [1/10] Batch [400/938] Loss: 0.3740 | Acc: 89.06%
Epoch [1/10] Batch [600/938] Loss: 0.2264 | Acc: 92.19%
Epoch [1/10] Batch [800/938] Loss: 0.2017 | Acc: 92.19%
Epoch [1/10] Batch [938/938] Loss: 0.1377 | Acc: 93.75%

=== Epoch 1 Summary ===
Train Loss: 0.5076 | Train Acc: 82.93%
Test  Loss: 0.1745 | Test  Acc: 94.55%

Epoch [2/10] Batch [1/938] Loss: 0.2260 | Acc: 90.62%
Epoch [2/10] Batch [200/938] Loss: 0.1442 | Acc: 96.88%
Epoch [2/10] Batch [400/938] Loss: 0.1866 | Acc: 93.75%
Epoch [2/10] Batch [600/938] Loss: 0.2654 | Acc: 92.19%
Epoch [2/10] Batch [800/938] Loss: 0.1883 | Acc: 95.31%
Epoch [2/10] Batch [938/938] Loss: 0.1466 | Acc: 93.75%

=== Epoch 2 Summary ===
Train Loss: 0.1993 | Train Acc: 93.87%
Test  Loss: 0.1427 | Test  Acc: 95.34%

Epoch [3/10] Batch [1/938] Loss: 0.1626 | Acc: 96.88%
Epoch [3/10] Batch [200/938] Loss: 0.1820 | Acc: 92